# Embedding Sequences with Attention

### Introduction:
Instead of getting embeddings from word local context, today we want to look at how to get embeddings from sequential text, and thus accounting word order:
- We start from looking at neural network
- Then application of neural network in NLP: feed forward neural network
- Finally the transformer, which is an imporvment over FNN that create word embeddings accounting word order (text sequence)

## Neural Network

### Basics of Neural Network
- Neural networks: deep learning models that solve machine learning problems, just like logistic regression or gradient boosted machines
- Advantages: 
    - Greatly outperform standard ML techniques on specialized problems, for example language modeling
- Downside:
    - Usually worse than standard ML on standard problems
    - Outputs are a black box and difficult to interpret
    - Models are often more challenging/labor-intensive to implement
    - Computational constraints: training requires specialized hardware

### Neuron
- Neuron is the most basic component of a neural network model
- It first applies dot product of a matrix of weight to a vector of numerical inputs:
    - Multiplies each input by a learned weight (parameter or coefficient)
    - Sums these products
- Then it applies a non-linear “activation function” to the sum
- Finally passes the output (a 1d number)

### Activation Functions
- Sigmoid function: 
    $$f(x) = \mathrm{sigmoid}(x)= \frac{1}{1 + \exp(-x)}$$
    - S-shaped function
    - Output between 1 and 0
- ReLU (rectified linear unit) function:
    $$g(x) = \mathrm{ReLU}(x) = \max\{0, x\}$$
    - More efficient than sigmoid and thus widely used in modern neural network models
    - Sigmoid does not work well in neural network, mainly because gradient is flat except around zero

### Multi-Layer Perceptron
- A multilayer perceptron (also called a feed-forward network or sequential model) stacks hidden layers vertically
- Hidden layer: a hidden layer is a component that stack multiple neurons together
    - Input: the input vector is the output vector from the last hidden layer / or input of the neural network, each neuron in the hidden layer face same input vector
    - Output: the output number from each neuron in the hidden layer is stacked together to form an output matrix
- The stack of mutiple hidden layers in the multi-layer perception is the reason why neural network models are generally called deep learning
- The end layer of a multi-Layer perceptron is the output layer, which is again a combination of several neurons
- The first layer of a multi-Layer perceptron is the input layer, which in general just pass the input vector to first hidden layer

### Equation Notation of Multi-Layer Perceptron
- An multi-layer perceptron (MLP) with two hidden layers can be expressed as: 
    $$y = g_2\big( g_1(x \cdot \omega_1) \cdot \omega_2 \big) \cdot \omega_y$$
    - where 
    $$y \in \{0,1\}^{n_y}, \quad x \in \mathbb{R}^{n_x}, \quad \omega_1 \in \mathbb{R}^{n_x \times n_1}, \quad \omega_2 \in \mathbb{R}^{n_1 \times n_2}, \quad \omega_y \in \mathbb{R}^{n_2 \times n_y}$$
    - $n_1$ and $n_2$ are dimensionality in first and second hidden layer
    - $n_y$ is the dimension of output
    - $ω_1$, $ω_2$, $ω_y$ are set of learnable weights for the first hidden, second hidden, and output layer
    - $g_1(·)$, $g_2(·)$ are element-wise non-linear functions (typically ReLU) for first and second layer
- The MLP can be also expressed in decomposed notation:
    - $h_1 = g_1(x \cdot \omega_1)$
    - $h_2 = g_2(h_1 \cdot \omega_2)$
    - $y = h_2 \cdot \omega_y$
- In most cases, the dimension of hidden layer < dimension of input; this is for information aggregation, the model should learn how to throw out some information that is irrelevant to the prediction problem

### Neural Network Training 
- Initialize Parameters: 
    - Set weights and biases for all layers, typically randomly
- Input Forward Propagation:
    - Pass input through layers
    - Apply activation functions
    - Derive output (prediction)
- Loss Calculation: compare predictions with true labels using a loss function
- Backward Propagation:
    - Compute gradients of the loss with respect to weights and biases through all layers via chain rule (from output to input)
- Weight Update: 
    - Adjust weights and biases using optimization algorithms (e.g., stochastic gradient descent)
- Repeat:
    - Iterate over the dataset for multiple epochs
    - Evaluate on validation data to monitor performance and prevent overfitting
- Stop:
    - End training based on stopping criteria (e.g., train loss convergence, maximum epochs)

### Neural Network Hyperparameter
- Epochs: Number of times the model trains over the full set of images
- Learning rate: How much weights/coefficients change in each update step (e.g., 0.01)
- Dropout rate: To avoid overfitting, a share of weights set to 0 (e.g., 0.5) at each iteration
- Train-validation ratio: split of data into two sets (e.g., 80-20 split)
- Loss function: How to evaluate model accuracy during training (e.g., Mean Squared Error)
- Activation functions: What type of nonlinear transformations (e.g., ReLU)
- More hyperparameters can be set (batch size etc.)

## Feedforward Neural Language Model


### Classic NLP Fails Sentence Classification Problem
- Consider a sentence classification problem: 
    - "I love this movie" --> positive
    - "I don't love this movei" --> negative
- Traditional NLP methods fail this task:
    - Bag-of-words model won’t capture the important difference between “don’t love” and “love”
        - For it, two sentence are similar as they both contain "love" and "movie"
        - Consider a even worse example: "There’s nothing I don’t love about this movie"
        - It does not capture word order and syntax (which word modifies what)
    - To fix this, some people use n-gram: making “don’t love” and “love” two different word phases, and thus allowing model to distinguish
        - However, N-grams with (4-grams or 3-grams) would result in a large feature space (large set of n-grams) 
        - It also doesn’t share information across similar n-grams: "don’t like" and "don’t love" treated completely different
- Feed forward neural network + sequence representations partically solve the issue (account word order), but with significant limitations
- Transformer fully solve the problem (account word order + allow word dependency across long text distance)

### Sequence Data and Neural Network
- FNN and later transformer models are called sequential deep learning for NLP
- Sequential deep learning for NLP use sequence data, which is the real breakthrough from classical NLP
    - Classical NLP: input = counts over words (bag-of-words)
    - Deep NLP: input = sequence of tokens $\{w_1,...,w_t,...w_n\}$, which preserves word order and structure
- Goal of sequential deep learning for NLP:
    - Predict the next word in a sequence: $P(w_t \mid w_1, \dots, w_{t-1})$
    - In practice, often approximated using a fixed context window: $P(w_t \mid w_{t-N+1}, \dots, w_{t-1})$
    - That is predict next word using previous $N-1$ words
- Sequential deep learning for NLP is self-supervised:
    - For word embedding, no need to label the data
    - Typically even no pre-processing (preserve the raw structure of text)

### Feedforward Neural Network Architecture
- Input: one-hot encoded word vectors from a fixed-size context window
    - Given a vocabulary size of V, each one-hot-encoded vector will be 1xV dimensional
    - One-hot-encoded vector: each word is represented by a vector such that only one element (which repsents the word) is 1, others are 0
    - Fixed-size context window: suppose context window is m, the model's inputs are one-hot-encoded vector representation of m words behind the word need to predict
- Embedding layer: 
    - The word embedding matrix (of size dxV) is mutiplied with each one-hot-encoded vector input to convert them into dense word embedding vector (of dimension d)
    - Value of elements in the word embedding matrix are weights that the neural network need to learn
    - Suppose a word is one-hot-encoded as a vector such that 5th element is 1, the word mebedding vector of the word is the 5th column of word embeding matrix
    - The m resulting embedding vectors are concatenated to produce e, the embedding layer output
    - Thus, the word order in the sequence text data is preserved through concatenated worder embedding vectors at the embedding layer
- Hidden layers: take the output of embeding layer as input, uses activation functions (ReLU)
- Output Layer: produces a probability distribution over possible next words using softmax
- Outcome: through back propagation and update on model weights, the word embedding matrix (and thus embedding vectors) will become better and better on capturing the semantic meaning of words

### Training Feedforward Neural Network
- Loss function: cross-entropy loss (negative log likelihood) for classification
    - Suppose $\hat y$ is a V dimensional vector denoting probability distribution over each word predicted by the model
    - $y$ is the empirical probability,  which is the one-hot-encoded vector of the true next word 
    - The cross entrophy is: $$L_{CE}(\hat{y}, y) = - \log \hat{y}_i = - \log p(w_t \mid w_{t-1}, \dots, w_{t-n+1})$$
    - Where $\hat{y}_i$ denotes the predicted probability that the correct word $i$ appears
- Optimization: 
    - Uses gradient descent and backpropagation to update parameters $θ$ (embedding matrix, weight matrices, and biases)
    $$ \theta_{s+1} = \theta_s - \eta \frac{\partial \left[ - \log p(w_t \mid w_{t-1}, \dots, w_{t-n+1}) \right]}{\partial \theta}$$

### Challenges of Feedforward Neural Network
- A small context window may fail to capture long-range dependencies and the full semantic meaning of a word
- Expanding the context window significantly increases the number of parameters and computational cost
- Many distant words in a large window may have weak or irrelevant relationships with the target word, introducing noise
- Requires large datasets to perform well
- All of these resulted in limited performance

## Transformer Model

### Extensions of Feedforward Neural Network 
Neural language models have evolved to address limitations of feedforward networks:
- Recurrent Neural Networks (RNNs):
    - Introduce recurrence (hidden states) to handle sequential data
    - Capture long-range dependencies, but suffer from vanishing gradients
- Convolutional Neural Networks (CNNs):
    - Use convolutional filters to extract local features
    - Efficient due to parallel processing; struggles with long-range dependencies
- Transformer Models
    - Use self-attention for global context
    - State-of-the-art in text data analysis
    - Examples: BERT, GPT

### Advantages of Transformer
- Recurrent neural nets can process whole documents word-by-word, but have to sweep through the whole document at each training epoch (sequential)
- Therefore, the training of recurrent neural nets are very slow 
- Transformers overcome these limitations: they provide a way to efficiently read in an entire document and learn the meaning of all words and all interactions between words (parallel)

### Architecture of Transformer
- Consider a sequence of tokens with fixed context length $n_L$, $\{w_1,...,w_i,...,w_{n_L}\}$
- After mutiplying with the embedding matrix, we have word embedding vectors $x_i = E(w_i)$ with dimension n$_E$ , producing a sequence of embedding vectors
- In previous models, the sequence $x1:n_L$ could be flattened to an $n_Ln_E$ dimensional vector and piped to the hidden layers for use in a task
- In transformers, a self-attention layer transforms $x1:n_L$ into a second sequence $h1:n_L$, this sequence is called attention-transformed vectors sequence
    - where: $$h_i = \sum_{j=1}^{n_L} a(x_i, x_j)\, x_j$$
    - and the attention function $a(.)$ has the porperty: $$a(\cdot) \ge 0, \quad \sum_{j=1}^{n_L} a(\cdot) = 1$$
    - thus, each $h_i$ is a weighted average of the whole sequence of embedding vectors of words in the context window
- $h1:n_L$ is flattened and piped to the network’s hidden layers, rather than $x1:n_L$

### Basic Self-Attention
- Basic self-attention function specifies:
    $$a(x_i, x_j) = \frac{\exp(x_i \cdot x_j)} {\sum_{k=1}^{n_L} \exp(x_i \cdot x_k)}$$
- The dot-product of embedding vectors $x_i· x_j$ is normalized with softmax such that $\sum_ja(·) = 1$
- The self-attention transformed vector of word i thus is: 
    $$h_i=\sum_{j=1}^{n_L} \frac{\exp(x_i \cdot x_j)x_j}{\sum_{k=1}^{n_L} \exp(x_i \cdot x_k)}\,$$
- This basic self-attention, though simple, can still be powerful:
    - It uses dot product to indirectly allow the word embeddings to interact with each other
    - The contribution from interaction between words to the prediction task would be stronger if two words embedding vectors are more similar
    - Thus, embedding layer will learn vectors that tend to have attention dot products that contribute to the task at hand
    - For example, in a sentiment classfication task, stopwords like “the” will not be helpful
    - The learned embedding $x_{the}$ will tend to have a low or negative dot product with more informative words
- However, this basic self-attention has no learnable parameters and ignores word order:
    - Successful models (e.g. BERT, GPT) do add parameters and word order information to $a(·)$

### Autoregressive vs Autoencoding Language Models
- Autoregressive models:
    - Example: GPT, “Generative Pre-Trained Transformer”
    - Pretrained on classic language modeling task: guess the next token having read all the previous ones
    - During training, attention heads only view previous tokens, not subsequent tokens
    - Ideal for text generation
- Autoencoding models:
    - Example: BERT, “Bidirectional Encoder Representations from Transformers”
    - Pretrained by dropping/shuffling input tokens and trying to reconstruct the original sequence
    - Usually build bidirectional representations and get access to the full sequence
    - Can be fine-tuned and achieve great results on many tasks, e.g. text classification